# Free particle in an oblique hard-wall box

**Status:** Maintained package-backed numerical laboratory for a nondimensional homogeneous-Dirichlet particle in a parallelogram.

**Prerequisite:** [Oblique hard-wall free particle](../../../../../docs/lecturenotes/qm/piab2d/tise/numerical/oblique-dirichlet-free-particle/index.md). Compare the distinct [periodic oblique-cell laboratory](oblique-periodic-free-particle.ipynb).

We set $\hbar=m=1$. The package retains only interior unknowns, applies the complete inverse metric including its mixed derivative, and imposes zero values on all four physical edges.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from projectkoios.physkit.qm.piab2d.oblique import ObliquePiab2D
from projectkoios.physkit.qm.piab2d.tise.finite_difference import ObliquePiab2DTiseFdSolveRequest, ObliquePiab2DTiseFdSolver
from projectkoios.physkit.units import MatrixQuantity, ScalarQuantity, Unitless, UnitSystem

direct_basis = np.array([[1.0, 0.45], [0.0, 0.90]])
model = ObliquePiab2D(
    direct_basis=MatrixQuantity(direct_basis, Unitless()),
    mass=ScalarQuantity(1.0, Unitless()),
    unit_system=UnitSystem.NONDIMENSIONAL,
)
request = ObliquePiab2DTiseFdSolveRequest(
    model=model, point_counts=(35, 37), state_count=6,
)
solution = ObliquePiab2DTiseFdSolver().action(request=request)
print("inverse metric =\n", model.metric.inverse_metric_tensor)
print("lowest energies =", solution.energies.magnitude)


## Representation checks

The sparse Hamiltonian must be Hermitian. Its states are normalized with the constant physical quadrature weight $|\det A|h_1h_2$, not with an unweighted sum or a periodic-cell point weight. Unlike the periodic model, this bounded problem has no constant zero-energy state.


In [ ]:
hamiltonian = solution.represented_hamiltonian.to_csr()
hermitian_residual = hamiltonian - hamiltonian.T
assert hermitian_residual.nnz == 0 or np.max(np.abs(hermitian_residual.data)) < 1e-12
represented_norms = (
    np.sum(solution.eigenfunctions.magnitude**2, axis=0)
    * solution.area_weight.magnitude
)
assert np.allclose(represented_norms, 1.0)
assert np.all(solution.energies.magnitude > 0.0)
assert model.metric.inverse_metric_tensor[0, 1] != 0.0
print(f"interior dimension = {request.interior_point_count}")
print(f"stored nonzeros = {solution.represented_hamiltonian.nonzero_count}")
print(f"area weight = {solution.area_weight.magnitude:.8e}")


## States on the physical parallelogram

The represented vectors omit their known zero boundary values. Restoring those values before mapping $\mathbf r=A\mathbf s$ makes the hard walls explicit and avoids displaying the fractional square as though it were the physical domain.


In [ ]:
def full_grid_state(interior_state, point_counts):
    """Restore zero Dirichlet boundaries around one interior state.

    Parameters
    ----------
    interior_state
        C-order interior samples.
    point_counts
        Full-grid extents including both boundaries.

    Returns
    -------
    numpy.ndarray
        Full grid with zero values on all four edges.
    """
    values = np.zeros(point_counts, dtype=np.float64)
    values[1:-1, 1:-1] = interior_state.reshape(
        (point_counts[0] - 2, point_counts[1] - 2), order="C",
    )
    return values

fractional_axes = tuple(np.linspace(0.0, 1.0, count) for count in request.point_counts)
fractional_0, fractional_1 = np.meshgrid(*fractional_axes, indexing="ij")
fractional_grid = np.stack((fractional_0, fractional_1), axis=-1)
cartesian_grid = fractional_grid @ direct_basis.T
fig, axes = plt.subplots(2, 2, figsize=(10.5, 8.0), constrained_layout=True)
for state_index, ax in enumerate(axes.flat):
    state = full_grid_state(
        solution.eigenfunctions.magnitude[:, state_index], request.point_counts,
    )
    density = state**2
    image = ax.pcolormesh(
        cartesian_grid[..., 0], cartesian_grid[..., 1], density,
        shading="gouraud", cmap="viridis",
    )
    ax.set(
        xlabel="x", ylabel="y", aspect="equal",
        title=f"state {state_index}, E={solution.energies.magnitude[state_index]:.5f}",
    )
    fig.colorbar(image, ax=ax, label=r"$|\psi|^2$")


## Orthogonal-limit benchmark and refinement

For $A=\operatorname{diag}(L_x,L_y)$, the exact continuum ground energy is

$$E_{11}=\frac{\pi^2}{2}\left(L_x^{-2}+L_y^{-2}\right).$$

This limit provides an independent reference for the same bounded finite-difference contract.


In [ ]:
length_x, length_y = 1.4, 0.8
rectangular_model = ObliquePiab2D(
    direct_basis=MatrixQuantity(np.diag([length_x, length_y]), Unitless()),
    mass=ScalarQuantity(1.0, Unitless()),
    unit_system=UnitSystem.NONDIMENSIONAL,
)
exact_ground_energy = 0.5 * np.pi**2 * (length_x**-2 + length_y**-2)
extents = np.array([9, 17, 33])
errors = []
for extent in extents:
    represented = ObliquePiab2DTiseFdSolver().action(
        request=ObliquePiab2DTiseFdSolveRequest(
            model=rectangular_model,
            point_counts=(int(extent), int(extent)),
            state_count=1,
        )
    )
    errors.append(abs(represented.energies.magnitude[0] - exact_ground_energy))
errors = np.asarray(errors)
ratios = errors[:-1] / errors[1:]
assert np.all((ratios > 3.9) & (ratios < 4.1))
fig, ax = plt.subplots(figsize=(6.4, 4.2))
ax.loglog(extents, errors, marker="o", label="observed")
ax.loglog(extents, errors[0] * (extents[0] / extents) ** 2, linestyle="--", label=r"$N^{-2}$ reference")
ax.set(xlabel="Full-grid points per axis", ylabel="Ground-energy error", title="Rectangular-limit second-order refinement")
ax.grid(which="both", alpha=0.25)
ax.legend()
fig.tight_layout()


## Interpretation and exercises

The rectangular continuum comparison verifies the zero-skew limit. Sparse Hermiticity and area normalization check the represented skew problem. The model covers a free scalar particle with hard-wall parallelogram boundaries.

1. Reduce the skew continuously to zero and compare the changing low spectrum.
2. Rotate both direct-basis columns together and verify spectral invariance.
3. Refine the two fractional axes independently and identify which spacing controls each low-state error.
4. Compare the ground-state density with the uniform periodic-cell plane wave and explain the boundary-driven difference.
5. Construct a physical-unit model and identify the wavefunction, area-weight, and energy units returned by the package.
